# Rule-Consistency Auditor
- Francesco Buda, francesco.buda3@studio.unibo.it
- Emanuele Sanchi, emanuele.sanchi@studio.unibo.it
- Tommaso Severi, tommaso.severi2@studio.unibo.it

## Import libraries

In [ ]:
# required for autoreload to work in Jupyter notebooks
%load_ext autoreload
%autoreload 2

In [ ]:
import utils.parsing_utils as pu
import utils.carla_utils as cu
from utils.violation_logger import ViolationLogger
from controller.state_machine import StateMachine
from models.state import StateType
from view.pygame_display import PygameDisplay
from carla_bindings import DataBinder

## Setup log file and CARLA world

In [ ]:
logger = ViolationLogger()
world, spectator, client = cu.world_connect(map_name="Town03")

In [ ]:
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05 
world.apply_settings(settings)
settings = world.get_settings()

## Spawn vehicles

In [ ]:
AUTOPILOT_VEHICLE_COUNT = 20
for i in range(AUTOPILOT_VEHICLE_COUNT):
    cu.spawn_random_vehicle_no_bike(world, spawn_index=i, autopilot=True)
ego_vehicle = cu.spawn_vehicle(world, spawn_index=10, autopilot=False)

## Setup data binder and state machines

In [ ]:
binder = DataBinder(world, ego_vehicle)
state_machines = [StateMachine(rule) for rule in pu.parse_rule_files("admin/rules/")]

## Main control loop

In [ ]:
from view.hud_drawer import HudDrawer, Notification

display = PygameDisplay(world, ego_vehicle)
hudDrawer = HudDrawer(display)
display.hud_drawer = hudDrawer
display.start()

print(f"-> Joystick detected by sub-process: {display._joystick is not None}")

frame_count = 0
RULE_CHECK = 10
try:
    while display.running:
        world.tick()
        if not display.tick():
            break
        ego_vehicle.apply_control(display.control)
        
        scene_data = binder.compute_scene_data()
        
        for sm in state_machines:
            current_state, transition = sm.evaluate(scene_data)
            
            if current_state != None:                
                if current_state.type == StateType.VIOLATION or current_state.type == StateType.WARNING:
                    hudDrawer.notify(Notification(sm.rule.name, current_state.type, duration=3.0))
                    if current_state.type == StateType.VIOLATION:
                        logger.log_violation(sm.rule, transition, scene_data.get_cached_dict())
        #print(ego_vehicle.is_at_traffic_light())
        print(f"in_intersection: {scene_data['in_intersection']},\n right_wedge_far: {scene_data['right_wedge_far']},\n right_wedge_near: {scene_data['right_wedge_near']}\n\n")
        #print(f"ego_location: {scene_data['ego_location']},\n in_intersection: {scene_data['in_intersection']},\n ego_can_enter_intersection: {scene_data['ego_can_enter_intersection']},\n other_can_enter_intersection: {scene_data['other_can_enter_intersection']}\n\n")
finally:

    logger.flush()
    display.destroy()

    print(f"Simulation ended after {frame_count} frames. Log: {logger.log_file}")

## Cleanup

In [ ]:
print("Destroying all vehicles in the simulator...")
for actor in world.get_actors().filter('vehicle.*'):
    try:
        actor.destroy()
        print(f"  Destroyed: {actor.id}")
    except Exception as e:
        print(f"  Skip {actor.id}: {e}")
print("All vehicles cleaned up. Ready for next run!")